In [ ]:

#scanning strategy but with hatching lines going up and down
import cadquery as cq
import math
import pandas as pd


hatch_spacing = 0.3
rotation_increment = 67
hatch_power, hatch_velocity = 250, 1000

print(f"Calculating smart toolpaths for {len(slices)} layers (Simulation Mode: Unfiltered)...")
master_data = []

for i, layer_wp in enumerate(slices):
    current_layer_num = i + 1
    current_angle = (i * rotation_increment) % 360
    
    wire_geom = layer_wp.val()
    z_height = wire_geom.Center().z
    flat_wire = wire_geom.translate((0, 0, -z_height))
    edges = flat_wire.Edges()

    if not edges: continue
    combined_wires = cq.Wire.combine(edges)
    if not combined_wires: continue
        
   
    combined_wires.sort(key=lambda w: w.BoundingBox().DiagonalLength, reverse=True)
    outer_wire = combined_wires[0]
    inner_wires = combined_wires[1:]

    slice_face = cq.Face.makeFromWires(outer_wire, inner_wires)

    bb = slice_face.BoundingBox()
    center_x, center_y = (bb.xmin + bb.xmax) / 2, (bb.ymin + bb.ymax) / 2
    diag = math.sqrt((bb.xmax - bb.xmin)**2 + (bb.ymax - bb.ymin)**2) * 1.5

    hatch_walls = (
        cq.Workplane("XY", origin=(center_x, center_y, 0))
        .rarray(xSpacing=hatch_spacing, ySpacing=diag, 
                xCount=int(diag/hatch_spacing), yCount=1, center=True)
        .rect(0.01, diag).extrude(1.0)
        .rotate((center_x, center_y, 0), (center_x, center_y, 1), current_angle)
    )

    thick_slice = cq.Workplane("XY").add(slice_face).extrude(0.5)
    hatched_solid = thick_slice.intersect(hatch_walls)

    hatch_edges = hatched_solid.section().val().Edges() if hatched_solid.val() else []

    raw_hatches = []
    normal_angle = math.radians(current_angle)
    parallel_angle = math.radians(current_angle + 90)

    for edge in hatch_edges:
        sx, sy = edge.positionAt(0.0).x, edge.positionAt(0.0).y
        ex, ey = edge.positionAt(1.0).x, edge.positionAt(1.0).y
        
       
        mid_x, mid_y = (sx + ex) / 2, (sy + ey) / 2
        sweep_score = (mid_x * math.cos(normal_angle)) + (mid_y * math.sin(normal_angle))
        
        start_proj = (sx * math.cos(parallel_angle)) + (sy * math.sin(parallel_angle))
        end_proj = (ex * math.cos(parallel_angle)) + (ey * math.sin(parallel_angle))
        
    
        if start_proj > end_proj:
            sx, sy, ex, ey = ex, ey, sx, sy
            proj_score = end_proj
        else:
            proj_score = start_proj
            
    
        raw_hatches.append({
            'Start_X': sx, 'Start_Y': sy, 'End_X': ex, 'End_Y': ey,
            'sweep_score': sweep_score, 'proj_score': proj_score
        })

    raw_hatches.sort(key=lambda item: item['sweep_score'])


    clusters, current_cluster, current_score = [], [], None
    
    for row in raw_hatches:
        if current_score is None:
            current_cluster.append(row)
            current_score = row['sweep_score']
        else:
            if abs(row['sweep_score'] - current_score) < (hatch_spacing * 0.5):
                current_cluster.append(row)
            else:
                clusters.append(current_cluster)
                current_cluster = [row]
                current_score = row['sweep_score']
    if current_cluster: clusters.append(current_cluster)

    
    for idx, cluster in enumerate(clusters):
        if idx % 2 != 0:
            cluster.sort(key=lambda item: item['proj_score'], reverse=True)
            for row in cluster:
                row['Start_X'], row['Start_Y'], row['End_X'], row['End_Y'] = row['End_X'], row['End_Y'], row['Start_X'], row['Start_Y']
        else:
          
            cluster.sort(key=lambda item: item['proj_score'])
            
        
        for row in cluster:
            master_data.append([
                current_layer_num, "Hatch", 
                round(row['Start_X'], 5), round(row['Start_Y'], 5), round(z_height, 5),
                round(row['End_X'], 5), round(row['End_Y'], 5), round(z_height, 5),
                hatch_power, hatch_velocity
            ])



columns = ["Layer", "Line_Type", "Start_X", "Start_Y", "Start_Z", 
           "End_X", "End_Y", "End_Z", "Power_W", "Velocity_mm_s"]

df = pd.DataFrame(master_data, columns=columns)

csv_filename = "LPBF_Simulation_ToolpathData.csv"
df.to_csv(csv_filename, index=False)

